In [ ]:
"""
@author: Chioma Opara and ChatGPT
Last modified: Fri Feb 8 2026

Allows sampling of images from saved DiT-XL/2 checkpoint files
"""

### Setting up environment

In [1]:
# Clone the fast DiT repo
!git clone https://github.com/chuanyangjin/fast-DiT.git
%cd fast-DiT

# Install dependencies
!pip install diffusers timm accelerate --upgrade

Cloning into 'fast-DiT'...
remote: Enumerating objects: 103, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 103 (delta 42), reused 31 (delta 31), pack-reused 49 (from 2)
Receiving objects: 100% (103/103), 6.38 MiB | 7.75 MiB/s, done.
Resolving deltas: 100% (52/52), done.
/content/fast-DiT
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 128.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.1/509.1 kB 51.1 MB/s eta 0:00:00
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.37.1
    Uninstalling diffusers-0.37.1:
      Successfully uninstalled diffusers-0.37.1


In [2]:
# Mount drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Load Hugging Face token directly from environment variables for security
from google.colab import userdata
huggingface_token = userdata.get('HF_TOKEN')

# Hugging Face hub login
from huggingface_hub import login
login(token=huggingface_token)

###  Changes to be made to sample.py

In [4]:
with open('/content/fast-DiT/sample.py', 'r') as f:
    content = f.read()

# Fix 1 — ensure the CustomDataset class correctly encodes labels from 0-1101
# Fix 2 — load pretrained weights with shape mismatch handling
# Fix 3 — add resume loading logic

# Fix 1 - checkpoint loading
old = """    ckpt_path = args.ckpt or f"DiT-XL-2-{args.image_size}x{args.image_size}.pt"
    state_dict = find_model(ckpt_path)
    model.load_state_dict(state_dict)
    model.eval()  # important!"""

new = """    if args.ckpt is not None:
        checkpoint = torch.load(
            args.ckpt,
            map_location=lambda storage, loc: storage,
            weights_only=False
        )

        if "ema" in checkpoint:
            state_dict = checkpoint["ema"]
        elif "model" in checkpoint:
            state_dict = checkpoint["model"]
        else:
            state_dict = checkpoint

    else:
        ckpt_path = f"DiT-XL-2-{args.image_size}x{args.image_size}.pt"
        state_dict = find_model(ckpt_path)

    model.load_state_dict(state_dict)
    model.eval()"""

content = content.replace(old, new)

# Fix 2
old = """    # Labels to condition the model with (feel free to change):
    class_labels = [207, 360, 387, 974, 88, 979, 417, 279]"""

new = """    # Labels to condition the model with (feel free to change):
    class_labels = args.class_labels"""

content = content.replace(old, new)

# Fix 3
old = """    y_null = torch.tensor([1000] * n, device=device)"""

new = """    y_null = torch.tensor([args.num_classes] * n, device=device)"""

content = content.replace(old, new)

# Fix 4
old = """    # Save and display images:
    save_image(samples, "sample.png", nrow=4, normalize=True, value_range=(-1, 1))"""

new = """    # Save and display images:
    save_image(samples, args.output, nrow=5, normalize=True, value_range=(-1, 1))"""

content = content.replace(old, new)

# Fix 5
old = """    parser.add_argument("--num-classes", type=int, default=1000)"""

new = """    parser.add_argument("--num-classes", type=int, default=1102)
    parser.add_argument("--class-labels", type=int, nargs="+", default=[1000, 1001, 1002, 1003])
    parser.add_argument("--output", type=str, default="sample.png", help="Output filename for generated samples.")"""

content = content.replace(old, new)

with open('/content/fast-DiT/sample.py', 'w') as f:
    f.write(content)

print("All fixes applied")

All fixes applied


In [5]:
# run structured sampling
import os
import random

checkpoint_dir = "/content/drive/MyDrive/ImageNet_Project/results/007-DiT-XL-2/checkpoints"
output_dir = "/content/drive/MyDrive/ImageNet_Project/samples"

os.makedirs(output_dir, exist_ok=True)

checkpoints = {
    "5K": "0005000.pt",
    "10K": "0010000.pt",
    "15K": "0015000.pt",
    "20K": "0020000.pt",
}

sampling_steps = [50, 100, 250]

# ~25% below 1000, ~75% above 1000
below_1000 = random.sample(range(0, 1000), 3)
above_1000 = random.sample(range(1000, 1102), 7)
class_labels = below_1000 + above_1000

print("Class labels:", class_labels)

for step_name, ckpt_file in checkpoints.items():
    ckpt_path = os.path.join(checkpoint_dir, ckpt_file)

    for num_steps in sampling_steps:
        output_path = os.path.join(
            output_dir,
            f"sample_ckpt_{step_name}_steps_{num_steps}.png"
        )

        labels_str = " ".join(map(str, class_labels))

        print(f"Sampling checkpoint {step_name}, sampling steps {num_steps}")

        !python /content/fast-DiT/sample.py \
          --model DiT-XL/2 \
          --image-size 256 \
          --num-classes 1102 \
          --ckpt {ckpt_path} \
          --class-labels {labels_str} \
          --cfg-scale 4.0 \
          --num-sampling-steps {num_steps} \
          --output {output_path}

Class labels: [725, 185, 24, 1017, 1046, 1029, 1002, 1023, 1025, 1024]
Sampling checkpoint 5K, sampling steps 50
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
config.json: 100% 547/547 [00:00<00:00, 2.35MB/s]
diffusion_pytorch_model.safetensors: 100% 335M/335M [00:03<00:00, 104MB/s]
  0% 0/50 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting

In [5]:
# TEST RUN - run the sampling script
# !python /content/fast-DiT/sample.py \
#   --model DiT-XL/2 \
#   --image-size 256 \
#   --num-classes 1102 \
#   --ckpt /content/drive/MyDrive/ImageNet_Project/results/007-DiT-XL-2/checkpoints/0020000.pt \
#   --class-labels 1000 \
#   --cfg-scale 4.0 \
#   --num-sampling-steps 10 \
#   --output /content/drive/MyDrive/ImageNet_Project/samples/sample_0020000.png

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
  0% 0/10 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more detai

In [6]:
# Wait and verify that google drive has synced before deleting runtime
!ls -lh /content/drive/MyDrive/ImageNet_Project/samples

# also force a sync
!sync

total 15M
-rw------- 1 root root 1020K May  8 16:06 sample_ckpt_10K_steps_100.png
-rw------- 1 root root 1023K May  8 16:07 sample_ckpt_10K_steps_250.png
-rw------- 1 root root  1.1M May  8 16:06 sample_ckpt_10K_steps_50.png
-rw------- 1 root root  1.4M May  8 16:10 sample_ckpt_15K_steps_100.png
-rw------- 1 root root  1.4M May  8 16:11 sample_ckpt_15K_steps_250.png
-rw------- 1 root root  1.4M May  8 16:10 sample_ckpt_15K_steps_50.png
-rw------- 1 root root  1.2M May  8 16:14 sample_ckpt_20K_steps_100.png
-rw------- 1 root root  1.3M May  8 16:15 sample_ckpt_20K_steps_250.png
-rw------- 1 root root  1.3M May  8 16:14 sample_ckpt_20K_steps_50.png
-rw------- 1 root root  1.2M May  8 16:02 sample_ckpt_5K_steps_100.png
-rw------- 1 root root  1.2M May  8 16:03 sample_ckpt_5K_steps_250.png
-rw------- 1 root root  1.2M May  8 16:01 sample_ckpt_5K_steps_50.png
-rw------- 1 root root  129K May  8 15:48 test_sample_0020000.png


In [ ]:
# resets to original training script
# !cd /content/fast-DiT && git checkout sample.py

In [ ]:
# function to keep Colab browswer active to prevent session crashes (run in browser console)
# function KeepAlive() {
#   document.querySelector('#top-toolbar').click();
#   console.log('Keeping alive:', new Date());
# }
# setInterval(KeepAlive, 60000);